In [2]:
!pip install finesse

In [3]:
import finesse
import numpy as np
from finesse.analysis.actions import Xaxis
from itertools import combinations
import matplotlib.pyplot as plt


finesse.configure(plotting=True)

### Notes / Questions

- The paper shows 4 QPDs (POPA, POPB, REFLA and REFLB), but does not mention how the Gouy phase separation between them is realised. I have modelled this using a simple lens system.

- The focal lengths and propagation distances of the Gouy lens are assumed.

- The paper gives the final decoupled signals (α, β and γ), but does not describe how these coefficients are obtained. Here they are calculated from the inverse of the sensing matrix.

- The final detector combination is chosen based on the lowest condition number. I am not sure whether this is the intended selection criterion or whether a fixed detector combination is expected.



The optical layout (based on the reference paper):

Laser → EOM → REFL pick-off → PRM → POP pick-off → ITM → ETM

According to the paper,

- 2 QPDs (near-field and far-field) are placed in both the REFL and POP paths.

- A Gouy phase shift is introduced between the detectors using a lens system.

- For yaw-sensing used (`pdtype=xsplit`)

- I adjusted the lens parameters to get a Gouy phase between detector A and B.

In [4]:
Lprc = 1.0  # PRC length, m
Larm = 5.0  # Arm length, m

def build_angular_model(
    demod_phases,
    L_tap_refl=0.6,
    L_refl_total=1.2,
    L_tap_pop=0.5,
    # Gouy Phase Lens Parameters:
    f_refl_lens=1.00,   # focal length for REFLB lens
    L_refl_lens1=0.20,  # distance from BS_REFL_SPLIT to Lens
    L_refl_lens2=1.12,  # distance from Lens to REFLB detector
    f_pop_lens=1.00,    # focal length for POPB lens
    L_pop_lens1=0.30,   # distance from BS_POP_SPLIT to Lens
    L_pop_lens2=0.95    # distance from Lens to POPB detector
):
    assert 0 < L_tap_refl < L_refl_total
    assert 0 < L_tap_pop < Lprc

    model = finesse.Model()
    model.parse(f"""

        #laser + modulator
        l laser1 P=1
        s s1 laser1.p1 mod.p1 L=0.1
        modulator mod f=75M midx=0.1 order=3

        #REFL path
        s s_refl_in mod.p2 REFL_BS.p1 L={L_refl_total - L_tap_refl}
        bs REFL_BS R=0.5 T=0.5 alpha=45
        s s_refl_main REFL_BS.p2 PRM.p1 L={L_tap_refl}

        s s_refl_tap REFL_BS.p4 BS_REFL_SPLIT.p1 L=0.2
        bs BS_REFL_SPLIT R=0.5 T=0.5 alpha=45

        # REFLA: near-field
        s s_REFLA BS_REFL_SPLIT.p2 REFLA_node.p1 L=0.05
        nothing REFLA_node

        # REFLB: far-field (lens used for 90° Gouy phase shift)
        s s_REFLB1 BS_REFL_SPLIT.p3 lens_refl.p1 L={L_refl_lens1}
        lens lens_refl f={f_refl_lens}
        s s_REFLB2 lens_refl.p2 REFLB_node.p1 L={L_refl_lens2}
        nothing REFLB_node

        #PRM
        m PRM R=0.8999 T=0.1001 phi=0 xbeta=0

        #POP path
        s s_pop_in PRM.p2 POP_BS.p1 L={L_tap_pop}
        bs POP_BS R=0.5 T=0.5 alpha=45
        s s_pop_main POP_BS.p2 ITM.p1 L={Lprc - L_tap_pop}

        s s_pop_tap POP_BS.p3 BS_POP_SPLIT.p1 L=0.2
        bs BS_POP_SPLIT R=0.5 T=0.5 alpha=45

        # POPA: near-field
        s s_POPA BS_POP_SPLIT.p2 POPA_node.p1 L=0.05
        nothing POPA_node

        # POPB: far-field (lens used for 90° Gouy phase shift)
        s s_POPB1 BS_POP_SPLIT.p3 lens_pop.p1 L={L_pop_lens1}
        lens lens_pop f={f_pop_lens}
        s s_POPB2 lens_pop.p2 POPB_node.p1 L={L_pop_lens2}
        nothing POPB_node

        #arm cavity
        m ITM R=0.991 T=0.009 phi=0 Rc=11 xbeta=0
        s s_arm ITM.p2 ETM.p1 L={Larm}
        m ETM R=0.9999 T=0.0001 phi=0 Rc=13 xbeta=0

        #for beam trancing
        cav ARM ITM.p2.o
        modes(maxtem=1) #introduce high order mode

        #QPDS
        pd1 POPA  POPA_node.p1.i  75M {demod_phases['POPA']}  pdtype=xsplit
        pd1 POPB  POPB_node.p1.i  75M {demod_phases['POPB']}  pdtype=xsplit
        pd1 REFLA REFLA_node.p1.i 75M {demod_phases['REFLA']} pdtype=xsplit
        pd1 REFLB REFLB_node.p1.i 75M {demod_phases['REFLB']} pdtype=xsplit
    """)
    return model

ANGULAR_PORTS = ["POPA", "POPB", "REFLA", "REFLB"]
ANGULAR_DOFS  = ["PRM.xbeta", "ITM.xbeta", "ETM.xbeta"]
XBETA_SCAN = 1e-7 #mirror scan range (in rad)

#calculating slope near the operating point
def calculate_slope(x, signal, window=5):
    centre = len(x) // 2
    x_fit = x[centre-window : centre+window+1]
    y_fit = signal[centre-window : centre+window+1]
    slope, _ = np.polyfit(x_fit, y_fit, 1)
    return slope

#scanning the mirror and storing detector outputs
def scan_dof(model, dof, npts=200):
    return model.run(Xaxis(dof, "lin", -XBETA_SCAN, XBETA_SCAN, npts))

- For every phase, the angular response to PRM, ITM and ETM yaw is calculated.

- Similar to the length sensing matrix, the optimum demod phase is chosen by maximising the sensitivity of the desired DoF while minimising the response to the others.

- In this model, POP → PRM and REFL → ARM


In [5]:
#optimising the demod phase

demod_phases_scan = np.arange(0, 361, 5) #range of demod phase
target_slopes = {port: {dof: [] for dof in ANGULAR_DOFS} for port in ANGULAR_PORTS}

#finding detector response for every demod phase
for phase in demod_phases_scan:
    phases = {p: phase for p in ANGULAR_PORTS}
    model = build_angular_model(phases)

    #scanning each mirror's DoF
    for dof in ANGULAR_DOFS:
        out = scan_dof(model, dof)
        x = out.x[0]

        #calculating the error signal
        for port in ANGULAR_PORTS:
            target_slopes[port][dof].append(calculate_slope(x, out[port]))

for port in ANGULAR_PORTS:
    for dof in ANGULAR_DOFS:
        target_slopes[port][dof] = np.asarray(target_slopes[port][dof])


#finding the optimum demod phase for each detector
best_phase_ang = {}

for port in ANGULAR_PORTS:
    prm = np.abs(target_slopes[port]["PRM.xbeta"])
    itm = np.abs(target_slopes[port]["ITM.xbeta"])
    etm = np.abs(target_slopes[port]["ETM.xbeta"])
    arm = itm + etm

    #POP --> PRM and REFL --> ARM
    ratio = (prm / (arm + 1e-20)) if port.startswith("POP") else (arm / (prm + 1e-20))
    idx = np.nanargmax(ratio)
    best_phase_ang[port] = demod_phases_scan[idx]

print("Optimum demod phases:")
for port in ANGULAR_PORTS:
    print(f"{port:8s} : {best_phase_ang[port]}°")


Optimum demod phases:
POPA     : 275°
POPB     : 350°
REFLA    : 85°
REFLB    : 150°


####  Raw angular sensing matrix

- I scanned the yaw (`xbeta`) for PRM, ITM and ETM.
- The raw matrix is constructed by using the slope of error signal of each detector.



In [6]:
#raw angular matrix

#use the optimum demod phase
model_ang = build_angular_model(best_phase_ang)

#initialise the matrix
M_raw = np.zeros((len(ANGULAR_PORTS), len(ANGULAR_DOFS)))

#detector response to yaw motion
for j, dof in enumerate(ANGULAR_DOFS):
    out = scan_dof(model_ang, dof) #xbeta scan
    x = out.x[0]
    #slope of error signal of each detector
    for i, port in enumerate(ANGULAR_PORTS):
        M_raw[i, j] = calculate_slope(x, out[port])

print(f"{'':10s}{'PRM':>15s}{'ITM':>15s}{'ETM':>15s}")
for i, port in enumerate(ANGULAR_PORTS):
    print(f"{port:10s}" + "".join(f"{M_raw[i,j]:15.5e}" for j in range(3)))



                      PRM            ITM            ETM
POPA          1.23373e-02   -4.28864e-02    8.54456e-02
POPB          5.91594e-02   -2.60153e-01    3.81653e-01
REFLA         3.94053e-03   -3.87507e-02    8.64986e-02
REFLB         4.68348e-04    8.06809e-03   -8.95784e-02


- POPA and REFLA show similar responses to the mirror motions
- POPB and REFLB have different values and opposite sign compared to POPA and REFLA due to the additional Gouy phase
- we use this matrix to construct the SOFT/HARD basis

#### HARD/SOFT Eigenbasis

- The ITM and ETM responses are transformed into this eigenbasis using the cavity g-factors, as raw sensing matrix is based on the physical mirror motions

- The transformed sensing matrix is therefore expressed in the basis [SOFT, HARD, PRM]

In [8]:
#Hard mode and soft mode Eigenbasis

g1 = 1 - Larm / 11.0   # ITM g-factor
g2 = 1 - Larm / 13.0   # ETM g-factor

print(f"\ng1(ITM)={g1:.4f}, g2(ETM)={g2:.4f}, g1*g2={g1*g2:.4f} (stable if 0<g1g2<1)")

#angular coupling matrix
A = np.array([[g2, 1.0], [1.0, g1]])
eigvals, eigvecs = np.linalg.eig(A) #eigen values and eigenvectors
order = np.argsort(-np.abs(eigvals))
idx_hard, idx_soft = order[0], order[1]

#normalising the HARD and SOFT eigenvectors
hard_vec = eigvecs[:, idx_hard] / np.linalg.norm(eigvecs[:, idx_hard])
soft_vec = eigvecs[:, idx_soft] / np.linalg.norm(eigvecs[:, idx_soft])

#extracting the PRM, ITM and ETM columns from raw sensing matrix (from the above step!)
col_PRM, col_ITM, col_ETM = M_raw[:, 0], M_raw[:, 1], M_raw[:, 2]

#transforming ITM and ETM into the SOFT and HARD basis
col_soft = soft_vec[0]*col_ITM + soft_vec[1]*col_ETM
col_hard = hard_vec[0]*col_ITM + hard_vec[1]*col_ETM

#sensing matrix in eigenmode basis
M_eigenbasis = np.column_stack([col_soft, col_hard, col_PRM])   # [SOFT, HARD, PRM]

print(f"{'':10s}{'SOFT':>15s}{'HARD':>15s}{'PRM':>15s}")
for i, port in enumerate(ANGULAR_PORTS):
    print(f"{port:10s}" + "".join(f"{M_eigenbasis[i,j]:15.5e}" for j in range(3)))


g1(ITM)=0.5455, g2(ETM)=0.6154, g1*g2=0.3357 (stable if 0<g1g2<1)
                     SOFT           HARD            PRM
POPA          9.12564e-02    2.85036e-02    1.23373e-02
POPB          4.55257e-01    7.79701e-02    5.91594e-02
REFLA         8.91411e-02    3.22101e-02    3.94053e-03
REFLB        -7.00431e-02   -5.64211e-02    4.68348e-04


- According to the paper, out of the 4 available detectors, only 3 are chosen because we have only 3 DoFs (SOFT, HARD and PRM) that needs controlling.

- For each combination, the condition number of the sensing matrix is calculated, and the combination with the smallest condition number is selected.

- The selected matrix is then inverted.

In [9]:
#to choose three detectors to sense three angular DoFs

dof_names = ["SOFT", "HARD", "PRM"] #DoF in transformed basis
#initialising variables to store the best detector combination
best_subset, best_cond, best_Minv = None, np.inf, None

#testing each combination
for subset in combinations(range(4), 3):

    #extracting the corresponding 3×3 sensing matrix
    Msub = M_eigenbasis[list(subset), :]
    #skipping invalid matrices
    if np.isnan(Msub).any() or np.all(Msub == 0):
        continue
    cond = np.linalg.cond(Msub) #calculating condition number
    if not np.isnan(cond) and cond < best_cond:
        best_cond, best_subset, best_Minv = cond, subset, np.linalg.inv(Msub)

if best_subset is None:
    raise ValueError(f"M_eigenbasis is degenerate. Raw matrix was:{M_raw}")

chosen_ports = [ANGULAR_PORTS[i] for i in best_subset]
M_final = M_eigenbasis[list(best_subset), :] #final sensing matrix

print(f"3-detectors are: {chosen_ports}")
print(f"The condition number: {best_cond:.4g}")
print("\nFinal Matrix: ([SOFT, HARD, PRM] basis)")
print(f"{'':10s}" + "".join(f"{d:>15s}" for d in dof_names))
for i, port in enumerate(chosen_ports):
    print(f"{port:10s}" + "".join(f"{M_final[i,j]:15.5e}" for j in range(3)))

print("\nInverse Matrix:")
print(best_Minv)



3-detectors are: ['POPA', 'REFLA', 'REFLB']
The condition number: 32.5

Final Matrix: ([SOFT, HARD, PRM] basis)
                     SOFT           HARD            PRM
POPA          9.12564e-02    2.85036e-02    1.23373e-02
REFLA         8.91411e-02    3.22101e-02    3.94053e-03
REFLB        -7.00431e-02   -5.64211e-02    4.68348e-04

Inverse Matrix:
[[ -10.98776824   32.83329868   13.19317069]
 [  14.70603774  -41.97141351  -34.25541575]
 [ 128.3528817  -145.89193063  -18.44486213]]


- The inverse of this matrix provides the control matrix used to generate the SOFT, HARD and PRM angular error signals


#### Row operations

- The inverse of the selected sensing matrix is used to obtain the detector combinations


In [10]:
#using row operations

#augmented matrix [M|I]
aug_M = np.hstack([M_final, np.eye(3)])
for col in range(3):

    #selecting the pivot element
    piv = np.argmax(np.abs(aug_M[col:, col])) + col
    if piv != col:
        aug_M[[col, piv]] = aug_M[[piv, col]]
    #normalisation
    aug_M[col] = aug_M[col] / aug_M[col, col]
    for r in range(3):
        if r != col:
            aug_M[r] = aug_M[r] - aug_M[r, col]*aug_M[col]

#Extracting the inverse sensing matrix
Minv = aug_M[:, 3:]

print("Decoupled error-signal combinations (θ = Minv . S)")
for i, dof in enumerate(dof_names):
    terms = "  +  ".join(f"({Minv[i,j]:.4g})*{chosen_ports[j]}" for j in range(3))
    print(f"\nθ_{dof:5s} = {terms}")


print("\nDiagonal sensitivities")
print(f"{'DOF':<10}{'Sensitivity':>15}")
for i, dof in enumerate(dof_names):
    sensitivity = 1.0 / Minv[i, i]
    print(f"{dof:<10}{sensitivity:15.5e}")

Decoupled error-signal combinations (θ = Minv . S)

θ_SOFT  = (-10.99)*POPA  +  (32.83)*REFLA  +  (13.19)*REFLB

θ_HARD  = (14.71)*POPA  +  (-41.97)*REFLA  +  (-34.26)*REFLB

θ_PRM   = (128.4)*POPA  +  (-145.9)*REFLA  +  (-18.44)*REFLB

Diagonal sensitivities
DOF           Sensitivity
SOFT         -9.10103e-02
HARD         -2.38257e-02
PRM          -5.42156e-02


- The coefficients define the decoupled error signals.
- The diagonal sensitivities were calculated from the inverse matrix



**DOUBT!!** The paper states that the angular signals are decoupled using coefficients (α, β, γ), but it does not describe how these coefficients are obtained. Here I have been calculated directly from the inverse of the simulated sensing matrix.